# BPMN Assistant - Evaluation scorecard [Kaggle]

Scores a trained LoRA adapter against the frozen `data/eval/*` sets. Objective metrics for C2 (XSD-validity), C3 (defect recall / false-positive rate) and C7 (failing-rule agreement); lexical proxy for C1/C4/C6. Written per the `kaggle-training-notebook` skill.

## Attach as inputs
1. Your **SFT adapter** - either the Stage-1 notebook Output, or a Dataset containing `adapter_config.json` + `adapter_model.safetensors`.
2. A **Dataset with the eval files** (`eval_c1_qa.jsonl`, `eval_c2_gen.jsonl`, `eval_c3_defects.jsonl`, `eval_c4_narrate.jsonl`, `eval_c6_automation.jsonl`, `eval_c7_compliance.jsonl`) - upload `data/eval/`.

GPU T4 + Internet ON, then Run All. Both paths auto-detect under `/kaggle/input`.

In [1]:
# 1) Deps (single GPU; SpiffWorkflow+lxml for C2 XSD validation)
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q -U "transformers>=4.51" "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=2.20" "SpiffWorkflow>=3.0" "lxml>=5.0"
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF token loaded.")
except Exception:
    print("No HF_TOKEN secret - continuing unauthenticated.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 33.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 25.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.3/275.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 40.0 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.3 MB/s eta 0:00:00
CUDA: True Tesla T4
No HF_TOKEN secret - continuing unauthenticated.


In [2]:
# 2) Config + auto-detect adapter and eval sets
import os, glob, json
ENABLE_THINKING = False
LIMIT_PER_SET = None    # set e.g. 15 for a quick smoke run; None = full
SYSTEM_PROMPT = 'You are a BPMN 2.0 expert assistant. Answer precisely and follow BPMN 2.0 conventions. When asked to generate a diagram, output valid BPMN 2.0 XML. When asked to review a diagram, identify concrete issues and how to fix them.'
_ad = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
_ev = glob.glob("/kaggle/input/**/eval_c1_qa.jsonl", recursive=True)
assert _ad, "Adapter not found - attach the SFT adapter (notebook Output or Dataset)."
assert _ev, "Eval sets not found - attach a Dataset containing data/eval/*.jsonl."
ADAPTER = os.path.dirname(_ad[0]); EVAL_DIR = os.path.dirname(_ev[0])
BASE_MODEL = json.load(open(_ad[0]))["base_model_name_or_path"]   # read base from the adapter
print("BASE_MODEL =", BASE_MODEL); print("ADAPTER =", ADAPTER); print("EVAL_DIR =", EVAL_DIR)

BASE_MODEL = Qwen/Qwen2.5-3B-Instruct
ADAPTER = /kaggle/input/datasets/lalitamittal/adapter1
EVAL_DIR = /kaggle/input/datasets/lalitamittal/bpmn-eval


In [3]:
# 3) Load base (4-bit) + adapter, inference-ready
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0}, trust_remote_code=True)
model = PeftModel.from_pretrained(base, ADAPTER)
model.config.use_cache = True; model.eval()
def generate(prompt, n=512):
    txt = tok.apply_chat_template([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":prompt}],
                                  tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    ids = tok(txt, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n, do_sample=False, use_cache=True)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
print("loaded:", BASE_MODEL, "+ adapter")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded: Qwen/Qwen2.5-3B-Instruct + adapter


In [4]:
# 4) Metrics (objective for C2/C3/C7; lexical proxy for C1/C4/C6) - per TDD S9
import re
from collections import Counter
from lxml import etree
from SpiffWorkflow.bpmn.parser.BpmnParser import BpmnParser, BpmnValidator
XML_STARTS = ("<?xml", "<definitions", "<bpmn:definitions", "<bpmn:")
FENCE = re.compile(r"```(?:xml)?\s*(.*?)```", re.DOTALL)
W = re.compile(r"[a-z0-9]+")
def toks(s): return W.findall((s or '').lower())
def token_f1(p, r):
    a, b = toks(p), toks(r)
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    ov = sum((Counter(a) & Counter(b)).values())
    if not ov: return 0.0
    pr, rc = ov/len(a), ov/len(b); return 2*pr*rc/(pr+rc)
def extract_xml(t):
    for b in FENCE.findall(t or ''):
        if b.strip().startswith(XML_STARTS): return b.strip()
    s = (t or '').strip(); return s if s.startswith(XML_STARTS) else None
def metric_c2(row, pred):
    xml = extract_xml(pred)
    if xml is None: return {'score':0.0,'xsd_valid':False}
    try: etree.fromstring(xml.encode()); wf=True
    except Exception: wf=False
    sc=False
    if wf:
        try: BpmnParser(validator=BpmnValidator()).add_bpmn_xml(etree.ElementTree(etree.fromstring(xml.encode())), filename='r.bpmn'); sc=True
        except Exception: sc=False
    return {'score': 1.0 if sc else (0.5 if wf else 0.0), 'xsd_valid': sc}
C3KW = {'unlabeled_task':['label','name','unnamed','unlabel'],
        'disconnected_element':['unreachable','disconnect','no incoming','not connected','orphan'],
        'missing_start_event':['start event','no start','instantiat','cannot start'],
        'missing_end_event':['end event','no end','never ends'],
        'gateway_type_mismatch':['deadlock','synchroni','and-join','parallel join','xor','stuck'],
        'lack_of_synchronization':['synchroni','multiple token','uncontrolled','lack of sync','multiple completion'],
        'implicit_split':['implicit','multiple outgoing','two outgoing','uncontrolled split'],
        'multiple_start_events':['multiple start','more than one start','several start','two start'],
        'start_with_incoming':['start','incoming'], 'end_with_outgoing':['end','outgoing'],
        'duplicate_id':['duplicate','unique','same id','id is used']}
ISSUE = ['issue','problem','error','invalid','deadlock','missing','unreachable','duplicate','not connected','incorrect','wrong']
CLEAN = ['no issue','no issues','no structural issue','no problem','no defect','no error','well-formed','is correct','looks correct','none found']
def metric_c3(row, pred):
    dt = (row.get('meta') or {}).get('defect_type'); low = (pred or '').lower()
    if dt == 'none':
        clean = any(c in low for c in CLEAN); flag = (not clean) and any(w in low for w in ISSUE)
        return {'score': 0.0 if flag else 1.0, 'kind':'clean', 'fp':flag}
    eid = ((row.get('meta') or {}).get('defect_element_id') or '').lower()
    det = any(k in low for k in C3KW.get(dt,[])) or (eid and eid in low)
    return {'score': 1.0 if det else 0.0, 'kind':'defect', 'detected':det}
def metric_c7(row, pred):
    res = (row.get('meta') or {}).get('results') or []
    fails = [r for r in res if not r.get('passed')]
    if not fails: return {'score': token_f1(pred, row.get('output',''))}
    low=(pred or '').lower(); hit=0
    for r in fails:
        w=[x for x in toks(r.get('desc','')) if len(x)>4][:4]
        if any(x in low for x in w): hit+=1
    return {'score': hit/len(fails)}
def metric_proxy(row, pred): return {'score': token_f1(pred, row.get('output',''))}
METRICS = {'C1':metric_proxy,'C2':metric_c2,'C3':metric_c3,'C4':metric_proxy,'C6':metric_proxy,'C7':metric_c7}
MAXTOK = {'C2':768}

In [5]:
# 5) Run eval + scorecard
import json, os
FILES = {'C1':'eval_c1_qa.jsonl','C2':'eval_c2_gen.jsonl','C3':'eval_c3_defects.jsonl',
         'C4':'eval_c4_narrate.jsonl','C6':'eval_c6_automation.jsonl','C7':'eval_c7_compliance.jsonl'}
board = {}
for cap, fn in FILES.items():
    path = os.path.join(EVAL_DIR, fn)
    if not os.path.exists(path): board[cap]={'n':0}; continue
    rows = [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]
    if LIMIT_PER_SET: rows = rows[:LIMIT_PER_SET]
    mfn = METRICS[cap]; extra=[]
    for r in rows:
        prompt = r['instruction'] + (('\n\n'+r['input']) if r.get('input') else '')
        pred = generate(prompt, n=MAXTOK.get(cap, 400))
        extra.append(mfn(r, pred))
    agg = {'n': len(rows), 'score': round(sum(e['score'] for e in extra)/len(rows), 3)}
    if cap=='C2': agg['xsd_valid_rate'] = round(sum(1 for e in extra if e.get('xsd_valid'))/len(rows), 3)
    if cap=='C3':
        d=[e for e in extra if e.get('kind')=='defect']; c=[e for e in extra if e.get('kind')=='clean']
        agg['defect_recall'] = round(sum(1 for e in d if e.get('detected'))/len(d),3) if d else None
        agg['false_positive_rate'] = round(sum(1 for e in c if e.get('fp'))/len(c),3) if c else None
    board[cap]=agg; print(cap, agg)
json.dump(board, open('/kaggle/working/eval_scorecard.json','w'), indent=2)
print('\n===== SCORECARD =====')
for cap, a in board.items(): print(f"{cap}: {a}")
print('\nsaved -> /kaggle/working/eval_scorecard.json')

C1 {'n': 12, 'score': 0.344}
C2 {'n': 6, 'score': 0.667, 'xsd_valid_rate': 0.667}
C3 {'n': 58, 'score': 0.379, 'defect_recall': 0.365, 'false_positive_rate': 0.5}
C4 {'n': 17, 'score': 0.427}
C6 {'n': 17, 'score': 0.702}
C7 {'n': 17, 'score': 0.843}

===== SCORECARD =====
C1: {'n': 12, 'score': 0.344}
C2: {'n': 6, 'score': 0.667, 'xsd_valid_rate': 0.667}
C3: {'n': 58, 'score': 0.379, 'defect_recall': 0.365, 'false_positive_rate': 0.5}
C4: {'n': 17, 'score': 0.427}
C6: {'n': 17, 'score': 0.702}
C7: {'n': 17, 'score': 0.843}

saved -> /kaggle/working/eval_scorecard.json
